# 1) Importação de Bibliotecas

In [ ]:
import json
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [38]:
root = Path.cwd()
meta_data_dir = root.parent
data_dir = root.parent / "model" / "data"

# 2) SetUp de Dados

In [39]:
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
metadata = json.loads((meta_data_dir / "model" / "metadata.json").read_text())

target_column = metadata['target']

x_train = train.drop(columns=[target_column], axis=1)
y_train = train[target_column]
x_test = test.drop(columns=[target_column], axis=1)
y_test = test[target_column]

class_order = metadata["classes"]
print(f"Train: {x_train.shape}  |  Test: {x_test.shape}")
print(f"Classes ({len(class_order)}): {class_order}")
print(f"Distribuição no train:\n{y_train.value_counts()}")

Train: (98493, 42)  |  Test: (24624, 41)
Classes (12): ['ARP_poisioning', 'DDOS_Slowloris', 'DOS_SYN_Hping', 'MQTT_Publish', 'Metasploit_Brute_Force_SSH', 'NMAP_FIN_SCAN', 'NMAP_OS_DETECTION', 'NMAP_TCP_scan', 'NMAP_UDP_SCAN', 'NMAP_XMAS_TREE_SCAN', 'Thing_Speak', 'Wipro_bulb']
Distribuição no train:
Attack_type
DOS_SYN_Hping                 75727
Thing_Speak                    6486
ARP_poisioning                 6200
MQTT_Publish                   3317
NMAP_UDP_SCAN                  2072
NMAP_XMAS_TREE_SCAN            1608
NMAP_OS_DETECTION              1600
NMAP_TCP_scan                   802
DDOS_Slowloris                  427
Wipro_bulb                      202
Metasploit_Brute_Force_SSH       30
NMAP_FIN_SCAN                    22
Name: count, dtype: int64


# 3) Treinamento dos modelos

In [40]:
rf_model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced_subsample", n_jobs=-1)
rf_model.fit(x_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [42]:
y_test_pred = rf_model.predict(x_test)

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- service_radius


In [ ]:
metrics = {
    "accuracy_score": accuracy_score(y_test, y_test_pred),
    "balanced_accuracy_score": balanced_accuracy_score(y_test, y_test_pred),
    "macro_f1_score": f1_score(y_true=y_test, y_pred=y_test_pred, average='macro'),
    "weighted_f1_score": f1_score(y_true=y_test, y_pred=y_test_pred, average='weighted'),
}

print(metrics)

In [ ]:
report = classification_report(y_test, y_test_pred, zero_division=0, target_names=class_order)
print(report)

In [ ]:
cm = confusion_matrix(y_true=y_test, y_pred=y_test_pred, normalize="true")

fig, axes = plt.subplots(figsize=(12, 9))
sns.heatmap(
    cm, annot=True, ax=axes,
    xticklabels=class_order,
    yticklabels=class_order
)

plt.title(f"Matriz de Confusão ({target_column})")
axes.set_xlabel("Valor Predito")
axes.set_ylabel("Valor Real")
plt.show()

In [ ]:
df_importances = pd.Series(rf_model.feature_importances_, index=x_train.columns).sort_values(ascending=False)

print('Avaliação de Importâncias')
display(df_importances)
